In [ ]:
# let's first prepare tif annual tif files
!pip install rasterio

In [ ]:
!pip install rasterio
import os
import xarray as xr
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt
from rasterio.enums import Resampling

# Define Input and Output Directories
input_nc_folder = "/content/drive/MyDrive/CCRI/Air_pollution/Annual"
output_tif_folder = "/content/drive/MyDrive/CCRI/Air_pollution/Annual_tif"
os.makedirs(output_tif_folder, exist_ok=True)
varname = "GWRPM25"

# Iterate through all NetCDF files in the folder
nc_files = [f for f in os.listdir(input_nc_folder) if f.endswith(".nc")]

for nc_file in nc_files:
    input_nc_path = os.path.join(input_nc_folder, nc_file)
    output_tif_filename = nc_file.replace(".nc", ".tif")
    output_tif_path = os.path.join(output_tif_folder, output_tif_filename)

    # Open NetCDF file
    ds = xr.open_dataset(input_nc_path)

    # Extract the variable as a raster (assumes a 2D array, or select a time slice if needed)
    data = ds[varname].values

    # Flip data vertically if latitudes are in ascending order
    # This makes the first row the northernmost.
    data = data[::-1, :]

    # Get geospatial metadata
    lat = ds["lat"].values
    lon = ds["lon"].values
    # from_origin expects (west, north, xsize, ysize)
    transform = from_origin(lon.min(), lat.max(), abs(lon[1] - lon[0]), abs(lat[1] - lat[0]))

    # Save as a Cloud Optimized GeoTIFF (COG)
    with rasterio.open(
        output_tif_path,
        "w",
        driver="GTiff",
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data.dtype,
        crs="EPSG:4326",
        transform=transform,
        tiled=True,             # Enable tiling
        blockxsize=256,         # Tile width (adjust as needed)
        blockysize=256,         # Tile height (adjust as needed)
        compress="deflate"      # Compression method (deflate is common for COGs)
    ) as dst:
        dst.write(data, 1)

        # Build internal overviews (pyramids) for efficient display at multiple scales
        overview_levels = [2, 4, 8, 16]
        dst.build_overviews(overview_levels, Resampling.nearest)
        dst.update_tags(ns='rio_overview', resampling='nearest')

        print(f"GeoTIFF saved at: {output_tif_path}")


GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/V5GL0502.HybridPM25.Global.200201-200212.tif
GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/V5GL0502.HybridPM25.Global.200301-200312.tif
GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/V5GL0502.HybridPM25.Global.200401-200412.tif
GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/V5GL0502.HybridPM25.Global.200001-200012.tif
GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/V5GL0502.HybridPM25.Global.199901-199912.tif
GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/V5GL0502.HybridPM25.Global.199801-199812.tif
GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/V5GL0502.HybridPM25.Global.200501-200512.tif
GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/V5GL0502.HybridPM25.Global.200101-200112.tif
GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/V

In [ ]:
!apt update && apt install -y gdal-bin python3-gdal
!pip install rasterio

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,243 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,788 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [75.2 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 https://developer.download.nvidia.com

In [ ]:
import os
import numpy as np
import rasterio
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.windows import Window
import subprocess

# Define directories
input_tif_folder = "/content/drive/MyDrive/CCRI/Air_pollution/Annual_tif"
output_vrt_path = os.path.join(input_tif_folder, "stacked_data.vrt")
tile_output_folder = os.path.join(input_tif_folder, "tiled_outputs")
os.makedirs(tile_output_folder, exist_ok=True)

# Step 1: Create VRT file of stacked TIFFs
print("Creating VRT file...")
subprocess.run([
    "gdalbuildvrt", "-separate", output_vrt_path
] + [os.path.join(input_tif_folder, file) for file in os.listdir(input_tif_folder) if file.endswith(".tif")])
print(f"VRT file created: {output_vrt_path}")


Creating VRT file...
VRT file created: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/stacked_data.vrt


In [ ]:
import os
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
import numpy as np

# Example input VRT path
input_vrt_path = "/content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/stacked_data.vrt"

# Output file path
output_tif_path = "/content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/pm25_p90_1998_2023.tif"

block_size = 1000  # Process data in 1000x1000 pixel blocks
last_n_years = 10  # Number of recent years to analyze

with rasterio.open(input_vrt_path) as src:
    height = src.height
    width = src.width
    num_years = src.count  # Each band represents a year

    # Ensure we have at least 10 years of data
    num_bands_to_use = min(last_n_years, num_years)
    start_band = num_years - num_bands_to_use + 1

    # Prepare metadata for output and remove conflicting keys
    meta = src.meta.copy()
    for key in ["driver", "width", "height", "transform", "crs"]:
        meta.pop(key, None)
    meta.update({
        "count": 1,
        "dtype": "float32"
    })

    # Create the output file with COG-friendly options
    with rasterio.open(
        output_tif_path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        crs=src.crs,
        transform=src.transform,
        tiled=True,
        blockxsize=256,
        blockysize=256,
        compress="deflate",
        **meta
    ) as dst:
        # Process in windows to save memory
        for row in range(0, height, block_size):
            for col in range(0, width, block_size):
                win_height = min(block_size, height - row)
                win_width = min(block_size, width - col)
                window = Window(col_off=col, row_off=row, width=win_width, height=win_height)

                # Read last `num_bands_to_use` bands in the window
                data = src.read(list(range(start_band, num_years + 1)), window=window)

                # Compute the 90th percentile across time, ignoring NaNs
                p90_recent_10_years = np.nanpercentile(data, 90, axis=0).astype(np.float32)

                # Write to output
                dst.write(p90_recent_10_years, 1, window=window)

        # Build overviews for faster display
        overview_levels = [2, 4, 8, 16]
        dst.build_overviews(overview_levels, Resampling.nearest)
        dst.update_tags(ns="rio_overview", resampling="nearest")

print(f"90th percentile GeoTIFF saved at: {output_tif_path}")


/usr/local/lib/python3.11/dist-packages/numpy/lib/_nanfunctions_impl.py:1634: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


90th percentile GeoTIFF saved at: /content/drive/MyDrive/CCRI/Air_pollution/Annual_tif/pm25_p90_1998_2023.tif
